# Aosta Valley Temperature Prediction — Data Exploration
Exploratory analysis of weather station data and GIS hydrological features.

In [ ]:
import sys
sys.path.append('..')

import matplotlib.pyplot as plt
import geopandas as gpd
from src.data.loader import load_config, load_weather_data, load_station_metadata, load_shapefiles
from src.data.preprocessor import build_pipeline

config = load_config('../config.yaml')

In [ ]:
weather_data = load_weather_data(f"../{config['data']['weather_data']}")
station_data = load_station_metadata(f"../{config['data']['station_data']}")
aosta_lakes, aosta_rivers = load_shapefiles(
    f"../{config['data']['lakes_shp']}",
    f"../{config['data']['rivers_shp']}"
)
print(weather_data.shape, station_data.shape)
weather_data.head()

## Weather Station Summary

In [ ]:
print(f"Stations: {station_data.shape[0]}")
print(f"Temperature readings: {weather_data.shape[0]}")
print(f"Date range: {weather_data['timestamp'].min()} → {weather_data['timestamp'].max()}")
weather_data['temperature'].describe()

## Temperature Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
weather_data['temperature'].dropna().hist(bins=50, ax=ax, color='steelblue', edgecolor='white')
ax.set_xlabel('Temperature (°C)')
ax.set_ylabel('Count')
ax.set_title('Temperature Distribution Across All Stations')
plt.tight_layout()
plt.show()

## Station Map

In [ ]:
station_gdf = gpd.GeoDataFrame(
    station_data,
    geometry=gpd.points_from_xy(station_data.longitude, station_data.latitude),
    crs="EPSG:4326"
)
lakes_plot = aosta_lakes.to_crs("EPSG:4326")
rivers_plot = aosta_rivers.to_crs("EPSG:4326")

fig, ax = plt.subplots(figsize=(10, 8))
rivers_plot.plot(ax=ax, color='cornflowerblue', linewidth=0.5, label='Rivers')
lakes_plot.plot(ax=ax, color='skyblue', label='Lakes')
station_gdf.plot(ax=ax, color='red', markersize=20, label='Weather Stations')
ax.set_title('Weather Stations, Lakes and Rivers — Aosta Valley')
ax.legend()
plt.tight_layout()
plt.show()

## Preprocessed Feature Preview

In [ ]:
station_gdf_full = build_pipeline(config, base_path='..')
print(f"Final dataset shape: {station_gdf_full.shape}")
station_gdf_full[['station_id', 'temperature', 'altitude', 'distance_to_lake', 'distance_to_river', 'hour', 'month']].head()